# Lab 22-02 · The Buck Converter in Steady State

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import buck as bk


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Ripple and RMS currents

*Predict* the inductor ripple, the output ripple and the RMS currents of the 12 V to 1.2 V, 10 A converter at 600 kHz with 0.6 µH and 62.5 µF.

In [2]:
VIN, V, I, FS, L, C = 12.0, 1.2, 10.0, 600e3, 0.6e-6, 62.5e-6
D = V / VIN
di = bk.ripple_current(V, D, FS, L)
print(f"ripple {di:.3f} A, output ripple {bk.ripple_voltage(di, FS, C)*1e3:.2f} mV")
print("RMS (L, HS, LS, Cin, Cout):", [round(x, 3) for x in bk.rms_currents(I, D, di)])
t, il, vo, pin, pout = bk.pss(VIN, D, FS, L, C, V / I)
print(f"exact: ripple {il.max()-il.min():.4f} A, output ripple {(vo.max()-vo.min())*1e3:.3f} mV")
s = ngspice("buck_ccm.cir")
print({k: s[k] for k in ("dia", "dva", "ilrmsa", "iqha")})
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(t * 1e6, il); ax[0].set_ylabel("i_L (A)")
ax[1].plot(t * 1e6, (vo - vo.mean()) * 1e3); ax[1].set_ylabel("v_out ripple (mV)"); ax[1].set_xlabel("time in period (us)")
plt.show()

ripple 3.000 A, output ripple 10.00 mV
RMS (L, HS, LS, Cin, Cout): [10.037, 3.174, 9.522, 3.012, 0.866]
exact: ripple 3.0017 A, output ripple 10.011 mV


{'dia': 3.000873, 'dva': 0.010008, 'ilrmsa': 10.0292, 'iqha': 3.1715}


## Part 2 · Conduction losses

*Predict* the duty cycle that holds 1.2 V at 10 A with 8, 3 and 1 mΩ, and the efficiency.

In [3]:
d4 = bk.duty_with_resistances(VIN, V, I, 8e-3, 3e-3, 1e-3)
req = d4 * 8e-3 + (1 - d4) * 3e-3 + 1e-3
print(f"D = {d4:.4f}, R_eq = {req*1e3:.3f} mohm, DC-model efficiency {V*I/(V*I + I*I*req)*100:.3f} %")
t, il, vo, pin, pout = bk.pss(VIN, d4, FS, L, C, V / I, 8e-3, 3e-3, 1e-3)
print(f"exact: output {np.mean(vo):.5f} V, efficiency {pout/pin*100:.3f} %")
print(f"ngspice: efficiency {s['etab']*100:.3f} %")
dd = bk.duty_with_diode(VIN, V, I, 0.5, 8e-3, 1e-3)
print(f"with a 0.5 V diode: D = {dd:.4f}, diode loss {(1-dd)*0.5*I:.2f} W")

D = 0.1038, R_eq = 4.519 mohm, DC-model efficiency 96.371 %
exact: output 1.20000 V, efficiency 96.343 %
ngspice: efficiency 96.346 %
with a 0.5 V diode: D = 0.1377, diode loss 4.31 W


## Part 3 · Discontinuous conduction

*Predict* the critical current, and the on-time that gives 1.2 V at 0.3 A.

In [4]:
print(f"critical current {bk.critical_current(V, D, FS, L):.2f} A")
K = bk.k_param(L, 4.0, FS)
d1 = bk.dcm_duty_for_ratio(V / VIN, K)
print(f"K = {K:.3f}, D1 = {d1:.4f}, on-time {d1/FS*1e9:.1f} ns, D2 = {d1*(VIN-V)/V:.4f}")
s2 = ngspice("buck_dcm.cir")
print({k: s2[k] for k in ("vavg", "ipk")})
iload = np.logspace(-2, 1.3, 300)
m = [bk.buck_ratio(0.1, bk.k_param(L, V / i, FS)) for i in iload]
plt.figure(); plt.semilogx(iload, m); plt.axvline(bk.critical_current(V, D, FS, L), ls=":", color="0.5")
plt.xlabel("load current at 1.2 V (A)"); plt.ylabel("M at fixed D = 0.1"); plt.show()

critical current 1.50 A
K = 0.180, D1 = 0.0447, on-time 74.5 ns, D2 = 0.4025


{'vavg': 1.194002, 'ipk': 1.343498}


## Part 4 · Pulse-frequency modulation

*Predict* the pulse rate of the wearable buck of Example 22-02.6 at 1 mA and at 1 µA.

In [5]:
ton = 1.8 / 3.6 * 556e-9
ipk, toff, q = bk.pfm_pulse(3.6, 1.8, ton, 2.2e-6)
print(f"on-time {ton*1e9:.0f} ns, peak {ipk:.4f} A, charge {q*1e9:.2f} nC")
for i in (1e-6, 1e-3, 0.1):
    print(f"I = {i*1e3:8.3f} mA: {i/q:12.1f} pulses per second")
print(f"PWM boundary at {q/556e-9:.4f} A")
i = np.logspace(-6, np.log10(0.114), 200)
plt.figure(); plt.loglog(i, i / q); plt.xlabel("load (A)"); plt.ylabel("pulse rate (Hz)"); plt.show()

on-time 278 ns, peak 0.2275 A, charge 63.23 nC
I =    0.001 mA:         15.8 pulses per second
I =    1.000 mA:      15814.7 pulses per second
I =  100.000 mA:    1581468.6 pulses per second
PWM boundary at 0.1137 A


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.